SEFIN DEUDA PUBLICA WEB SCRAPING ANUAL

In [ ]:
import requests
import pandas as pd
from bs4 import BeautifulSoup
import urllib3
urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

fechainicio = '2015-01-01'

url="https://www.sefin.gob.hn/deuda/"
res=requests.get(url,verify=False)
soup=BeautifulSoup(res.text, 'html.parser')

tabla=soup.find("table")
datos=[]

filas=tabla.find_all("tr")
headers=[th.get_text(strip=True) for th in filas[0].find_all("th")]

for fila in filas[1:]:
    celdas=[td.get_text(strip=True) for td in fila.find_all("td")]
    if celdas:  # Check if cols is not empty
        datos.append(celdas)

df_raw=pd.DataFrame(datos, columns=headers)
df=df_raw.rename(columns={df_raw.columns[1]:"NOMBRE_INDICADOR"})

columnas_años = [col for col in df.columns if col.isdigit()]
df_filtrado= df[["NOMBRE_INDICADOR"] + columnas_años]

df_largo=df_filtrado.melt(id_vars=["NOMBRE_INDICADOR"],var_name="ANIO",value_name="VALOR")
df_largo["VALOR"] = df_largo["VALOR"].str.replace(",", "").astype(float)  # Convert  float

df_largo["FECHA"] = pd.to_datetime(df_largo["ANIO"].astype(str)+ '-01-01')
df_largo.drop(columns=["ANIO"],inplace=True)

df_largo["TIPO"]="Deuda Publica"
df_largo["PERIODICIDAD"]="Anual"
df_largo["VARIACION"]=df_largo.groupby('NOMBRE_INDICADOR')['VALOR'].pct_change().round(4)
df_largo["DESCRIPCION"]="SEFIN"


df_filtrado=df_largo[df_largo["FECHA"]>=fechainicio]
df_filtrado.head(10)

,NOMBRE_INDICADOR,VALOR,FECHA,TIPO,PERIODICIDAD,VARIACION,DESCRIPCION
28,Deuda Interna,3461.9,2015-01-01,Deuda Publica,Anual,0.0621,SEFIN
29,Deuda Externa,5732.6,2015-01-01,Deuda Publica,Anual,0.0570,SEFIN
30,Deuda Total,9194.5,2015-01-01,Deuda Publica,Anual,0.0589,SEFIN
31,Deuda/PIB,44.7,2015-01-01,Deuda Publica,Anual,-0.0067,SEFIN
32,Deuda Interna,3929.4,2016-01-01,Deuda Publica,Anual,0.1350,SEFIN
33,Deuda Externa,5840.3,2016-01-01,Deuda Publica,Anual,0.0188,SEFIN
34,Deuda Total,9769.6,2016-01-01,Deuda Publica,Anual,0.0625,SEFIN
35,Deuda/PIB,46.3,2016-01-01,Deuda Publica,Anual,0.0358,SEFIN
36,Deuda Interna,4144.5,2017-01-01,Deuda Publica,Anual,0.0547,SEFIN
37,Deuda Externa,6780.1,2017-01-01,Deuda Publica,Anual,0.1609,SEFIN


In [4]:
################           INSERT TABLE / DATA           #################   #      
from azure.identity import InteractiveBrowserCredential
import pandas as pd
from Server import AZURE
from tqdm import tqdm
from sqlalchemy import create_engine, text

credential = InteractiveBrowserCredential()

server = AZURE
database = 'sqlpooldwhandr01'
schema = 'HN_NAP_HO_MISRIESGOS_F'
tabla = 'INDICADORES_MACROECONOMICOS'
driver = "ODBC Driver 17 for SQL Server"

connection_string = (
        f"DRIVER={driver};"
        f"SERVER={server};"
        f"DATABASE={database};"
        f"Encrypt=yes;"
        f"TrustServerCertificate=no;"
        f"Authentication=ActiveDirectoryInteractive;"
    )

connection_uri = f"mssql+pyodbc:///?odbc_connect={connection_string}"
engine = create_engine(connection_uri, fast_executemany=True)


query = text("""
SELECT COLUMN_NAME, DATA_TYPE , CHARACTER_MAXIMUM_LENGTH
FROM INFORMATION_SCHEMA.COLUMNS 
WHERE TABLE_NAME = :tablita
AND TABLE_SCHEMA = :esquema
""")

with engine.connect() as conn:
    result = conn.execute(query, {"tablita": tabla, "esquema":schema})
    columns_types = {row[0]: [row[1] , row[2]] for row in result}

In [5]:
#-----------------------------############### INSERT ##################------------------------------------#

#df_resultado['DATE_TIME']=pd.to_datetime(df_resultado['DATE_TIME']).dt.strftime('%Y-%m-%d')
data_frame=df_filtrado
chunksize = 100

for start in tqdm(range(0, len(data_frame), chunksize), desc="Insertando datos"):
    end = min(start + chunksize, len(data_frame))
    chunk = data_frame.iloc[start:end]

    try:
        chunk.to_sql(tabla, 
                     con=engine, 
                     schema=schema, 
                     if_exists='append', 
                     index=False, 
                     chunksize=chunksize)
    except Exception as e:
        print(f"Error al insertar datos en la base de datos: {e}")
    #break
print('Proceso de insercion completado')

Insertando datos: 100%|██████████| 1/1 [00:05<00:00,  5.14s/it]

Proceso de insercion completado
